<a href="https://colab.research.google.com/github/hyunkyung31/coronary-ai-ml-dl/blob/main/hyunkyung/03_RF_mock_%EA%B2%80%EC%A6%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


# 1단계: 최종 저장 폴더와 mock 생성에 필요한 파일 확인

In [2]:
from pathlib import Path

FINAL_RF_DIR = Path(
    "/content/drive/MyDrive/clinical_rf_validation/"
    "rf_nested_20260828_030434/"
    "final_rf_sigmoid_20260828_072752"
)

assert FINAL_RF_DIR.exists(), (
    f"최종 모델 폴더를 찾을 수 없습니다:\n{FINAL_RF_DIR}"
)

saved_files = sorted(
    path for path in FINAL_RF_DIR.rglob("*")
    if path.is_file()
)

print("✅ 최종 모델 폴더 확인")
print(f"저장 위치: {FINAL_RF_DIR}")
print(f"파일 개수: {len(saved_files)}\n")

for path in saved_files:
    relative_path = path.relative_to(FINAL_RF_DIR)
    file_size_kb = path.stat().st_size / 1024
    print(f"- {relative_path} ({file_size_kb:,.1f} KB)")

✅ 최종 모델 폴더 확인
저장 위치: /content/drive/MyDrive/clinical_rf_validation/rf_nested_20260828_030434/final_rf_sigmoid_20260828_072752
파일 개수: 14

- artifact_manifest.json (0.4 KB)
- artifact_reference_predictions.npz (2.8 KB)
- calibration_comparison.csv (0.7 KB)
- calibration_metrics.json (1.3 KB)
- calibration_oof_predictions.csv (18.6 KB)
- calibration_oof_predictions.npz (6.8 KB)
- calibration_thresholds.json (0.1 KB)
- development_oof_calibration_curve.png (139.0 KB)
- final_model_metadata.json (2.6 KB)
- final_rf_candidate_selection.csv (12.2 KB)
- final_rf_params.json (0.2 KB)
- final_rf_sigmoid_model.joblib (408.3 KB)
- input_schema.csv (2.0 KB)
- new_patient_template.csv (0.5 KB)


# 2단계: 모델 무결성·threshold·입력 템플릿 확인

In [3]:
import json
import hashlib
import pandas as pd

# 파일 경로
MODEL_PATH = FINAL_RF_DIR / "final_rf_sigmoid_model.joblib"
METADATA_PATH = FINAL_RF_DIR / "final_model_metadata.json"
THRESHOLD_PATH = FINAL_RF_DIR / "calibration_thresholds.json"
SCHEMA_PATH = FINAL_RF_DIR / "input_schema.csv"
TEMPLATE_PATH = FINAL_RF_DIR / "new_patient_template.csv"

# 1. 저장 모델 SHA-256 재확인
with open(MODEL_PATH, "rb") as file:
    model_sha256 = hashlib.sha256(file.read()).hexdigest()

expected_sha256 = (
    "43fdf7f22dd7aed1d9d2338325b67018"
    "b9394d9a756ca75e88f141a595d451ac"
)

assert model_sha256 == expected_sha256, (
    "저장된 모델의 SHA-256이 기존 기록과 다릅니다."
)

# 2. 저장 설정 불러오기
with open(METADATA_PATH, "r", encoding="utf-8") as file:
    final_metadata = json.load(file)

with open(THRESHOLD_PATH, "r", encoding="utf-8") as file:
    calibration_thresholds = json.load(file)

input_schema = pd.read_csv(SCHEMA_PATH)
new_patient_template = pd.read_csv(TEMPLATE_PATH)

print("✅ 모델 SHA-256 일치")
print("SHA-256:", model_sha256)

print("\n✅ 저장 threshold")
print(calibration_thresholds)

print("\n✅ Metadata 주요 항목")
print("저장된 항목:", list(final_metadata.keys()))

print("\n✅ 입력 스키마")
print("shape:", input_schema.shape)
display(input_schema)

print("\n✅ 신규 환자 템플릿")
print("shape:", new_patient_template.shape)
display(new_patient_template)

✅ 모델 SHA-256 일치
SHA-256: 43fdf7f22dd7aed1d9d2338325b67018b9394d9a756ca75e88f141a595d451ac

✅ 저장 threshold
{'Raw RF': 0.41948487020824166, 'Sigmoid': 0.3558017639131723, 'Isotonic': 0.3333333333333333}

✅ Metadata 주요 항목
저장된 항목: ['ArtifactType', 'Model', 'Calibration', 'CalibrationCV', 'FinalRFParams', 'OperatingThreshold', 'ThresholdPolicy', 'SigmoidDevelopmentOOFMetrics', 'DevelopmentSampleCount', 'TargetDistribution', 'RawInputCount', 'TransformedInputCount', 'InputColumns', 'ExcludedColumns', 'DataSignature', 'NestedCVRunDirectory', 'ReferenceProbabilitySHA256', 'ImportantWarning', 'Environment']

✅ 입력 스키마
shape: (54, 7)


,Position,Column,VariableType,PandasDtype,AllowedValues,ObservedMin,ObservedMax
0,0,Age,numeric,int64,NaN,30.000000,86.000000
1,1,Weight,numeric,int64,NaN,48.000000,120.000000
2,2,Length,numeric,int64,NaN,140.000000,188.000000
3,3,Sex,binary,int64,0|1,0.000000,1.000000
4,4,BMI,numeric,float64,NaN,18.115413,40.900658
5,5,DM,binary,int64,0|1,0.000000,1.000000
6,6,HTN,binary,int64,0|1,0.000000,1.000000
7,7,Current Smoker,binary,int64,0|1,0.000000,1.000000
8,8,EX-Smoker,binary,int64,0|1,0.000000,1.000000
9,9,FH,binary,int64,0|1,0.000000,1.000000



✅ 신규 환자 템플릿
shape: (1, 54)


,Age,Weight,Length,Sex,BMI,DM,HTN,Current Smoker,EX-Smoker,FH,...,HB,K,Na,WBC,Lymph,Neut,PLT,EF-TTE,Region RWMA,VHD
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 3단계: 정상 mock 환자 3명 생성 및 스키마 검사